# CANDLE/FCIS - Causal Discovery

Discover causal relationships using PCMCI+ algorithm.

In [1]:
import sys
sys.path.append('../src')

from causal_discovery import CausalDiscoveryEngine
from visualization import CANDLEVisualizer
import pandas as pd
import matplotlib.pyplot as plt

# Initialize visualizer
viz = CANDLEVisualizer('../results/figures')

Tigramite not available: No module named 'dcor'. Using fallback implementation.


## 1. Load Processed Data

In [2]:
# Load merged data (prices + sentiment)
data = pd.read_csv('../data/processed/merged_data_normalized.csv', index_col=0, parse_dates=True)
print(f"Data shape: {data.shape}")
print(f"Variables: {list(data.columns)}")
data.head()

Data shape: (2665, 25)
Variables: ['ADANIENT', 'BAJFINANCE', 'HDFCBANK', 'ICICIBANK', 'INFY', 'MARUTI', 'RELIANCE', 'SBIN', 'TCS', 'WIPRO', 'sentiment_mean', 'sentiment_std', 'news_count', 'positive_prob_mean', 'negative_prob_mean', 'sentiment_momentum', 'sentiment_volatility', 'sentiment_lag1', 'sentiment_lag2', 'sentiment_lag3', 'sentiment_ma5', 'sentiment_ma10', 'extreme_positive', 'extreme_negative', 'sentiment_trend']


,ADANIENT,BAJFINANCE,HDFCBANK,ICICIBANK,INFY,MARUTI,RELIANCE,SBIN,TCS,WIPRO,...,sentiment_momentum,sentiment_volatility,sentiment_lag1,sentiment_lag2,sentiment_lag3,sentiment_ma5,sentiment_ma10,extreme_positive,extreme_negative,sentiment_trend
Date,,,,,,,,,,,,,,,,,,,,,
2016-01-01,-1.837120,-0.370124,-1.262690,-1.534977,-1.446313,-0.760837,-1.235620,-1.569910,-1.345801,0.115678,...,-1.336496,-3.806858,-1.454604,-0.478861,1.408499,0.339898,-1.602082,-0.60252,-0.706517,-0.213874
2016-01-02,-1.837120,-0.370124,-1.262690,-1.534977,-1.446313,-0.760837,-1.235620,-1.569910,-1.345801,0.115678,...,-1.336496,-3.806858,-0.895013,-0.478861,1.408499,0.339898,-1.602082,-0.60252,1.413035,-2.420459
2016-01-03,-1.837120,-0.370124,-1.262690,-1.534977,-1.446313,-0.760837,-1.235620,-1.569910,-1.345801,0.115678,...,-1.336496,-3.806858,-2.539950,0.352986,1.408499,0.339898,-1.602082,-0.60252,-0.706517,-2.018517
2016-01-04,-1.837120,-0.370124,-1.262690,-1.534977,-1.446313,-0.760837,-1.235620,-1.569910,-1.345801,0.115678,...,-2.701045,-3.806858,-2.240315,-2.092257,1.934159,0.339898,-1.602082,-0.60252,1.413035,-3.134319
2016-01-05,1.215949,-0.454698,-0.591968,0.208759,-0.285660,-0.206029,0.535002,-0.687245,-0.615831,-0.114732,...,-1.821809,-3.806858,-3.072109,-1.646841,0.388961,-3.651828,-1.602082,-0.60252,1.413035,-2.094933


## 2. Run PCMCI+ Causal Discovery

In [3]:
# Initialize engine
engine = CausalDiscoveryEngine(
    pc_alpha=0.05,      # Significance level
    tau_max=5,          # Max time lag
    tau_min=1,          # Min time lag
    independence_test='parcorr'
)

# Run discovery
results = engine.discover_causal_graph(data)

# Get graph
graph = results['graph']
print(f"Nodes: {len(graph.nodes)}")
print(f"Edges: {len(graph.edges)}")

Running in fallback mode with simplified algorithm
/Users/kshitizsikriwal/Kshitiz/causal-3/.venv/lib/python3.13/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


Nodes: 25
Edges: 211


## 3. Visualize Causal Graph

In [4]:
import pickle
import networkx as nx

# Save graph in modern format (GPickle deprecated in NetworkX 3.0)
# Use GraphML (human-readable XML format)
nx.write_graphml(graph, '../data/processed/causal_graph.graphml')

# Also save as Pickle manually (for Python objects)
with open('../data/processed/causal_graph.pkl', 'wb') as f:
    pickle.dump(graph, f)

# Save parents dict
parents_df = pd.DataFrame([
    {'variable': k, 'parents': str(v)}
    for k, v in parents.items()
])
parents_df.to_csv('../data/processed/causal_parents.csv', index=False)

print("✓ Graph saved to:")
print("  • data/processed/causal_graph.graphml (GraphML format)")
print("  • data/processed/causal_graph.pkl (Pickle format)")
print("  • data/processed/causal_parents.csv (CSV format)")

NameError: name 'parents' is not defined

## 4. Analyze Causal Parents

In [ ]:
# Get parents for each variable
parents = results['parents']

for var, var_parents in parents.items():
    if var_parents:
        print(f"\n{var} is caused by:")
        for parent, lag in var_parents:
            print(f"  - {parent} (lag {lag})")


ADANIENT is caused by:
  - INFY (lag 1)
  - INFY (lag 2)
  - INFY (lag 3)
  - INFY (lag 4)
  - INFY (lag 5)
  - sentiment_ma5 (lag 1)
  - extreme_negative (lag 5)

BAJFINANCE is caused by:
  - ICICIBANK (lag 3)
  - ICICIBANK (lag 4)
  - ICICIBANK (lag 5)
  - INFY (lag 1)
  - INFY (lag 2)
  - INFY (lag 3)
  - INFY (lag 4)
  - INFY (lag 5)
  - MARUTI (lag 3)
  - MARUTI (lag 4)
  - sentiment_ma10 (lag 3)
  - extreme_positive (lag 3)
  - extreme_positive (lag 4)

HDFCBANK is caused by:
  - ICICIBANK (lag 2)
  - ICICIBANK (lag 3)
  - ICICIBANK (lag 4)
  - ICICIBANK (lag 5)
  - INFY (lag 1)
  - INFY (lag 2)
  - MARUTI (lag 2)
  - MARUTI (lag 3)
  - MARUTI (lag 4)
  - MARUTI (lag 5)
  - RELIANCE (lag 3)
  - RELIANCE (lag 5)

ICICIBANK is caused by:
  - HDFCBANK (lag 5)
  - INFY (lag 1)
  - INFY (lag 2)
  - INFY (lag 3)
  - INFY (lag 4)
  - INFY (lag 5)
  - SBIN (lag 2)
  - sentiment_lag2 (lag 2)

INFY is caused by:
  - ICICIBANK (lag 1)
  - ICICIBANK (lag 2)
  - ICICIBANK (lag 3)
  - ICICIBA

## 5. Save Results

In [ ]:
import pickle
import networkx as nx

# Save graph in modern format (GPickle deprecated in NetworkX 3.0)
# Use GraphML (human-readable XML format)
nx.write_graphml(graph, '../data/processed/causal_graph.graphml')

# Also save as Pickle manually (for Python objects)
with open('../data/processed/causal_graph.pkl', 'wb') as f:
    pickle.dump(graph, f)

# Save parents dict
parents_df = pd.DataFrame([
    {'variable': k, 'parents': str(v)}
    for k, v in parents.items()
])
parents_df.to_csv('../data/processed/causal_parents.csv', index=False)

print("✓ Graph saved to:")
print("  • data/processed/causal_graph.graphml (GraphML format)")
print("  • data/processed/causal_graph.pkl (Pickle format)")
print("  • data/processed/causal_parents.csv (CSV format)")

✓ Graph saved to:
  • data/processed/causal_graph.graphml (GraphML format)
  • data/processed/causal_graph.pkl (Pickle format)
  • data/processed/causal_parents.csv (CSV format)
